# GenAI RAG Chatbot for House Price Prediction

This notebook builds a Retrieval-Augmented Generation chatbot using the cleaned Bengaluru house price dataset, saved regression model, embeddings, vector database, and Groq LLM.

In [1]:
import pandas as pd
import pickle

df = pd.read_csv("../Data/processed/clean_house_data.csv")

# later, load your trained model
# model = pickle.load(open("house_price_model.pkl", "rb"))

df.head()

,area_type,availability,location,size,total_sqft,bath,balcony,price,bhk,price_per_sqft,sqft_per_bhk
0,Super built-up Area,Ready To Move,1st Phase JP Nagar,3 BHK,1875.0,3.0,1.0,167.0,3,8906.666667,625.000000
1,Built-up Area,Ready To Move,1st Phase JP Nagar,5 Bedroom,1500.0,5.0,2.0,85.0,5,5666.666667,300.000000
2,Super built-up Area,Ready To Move,1st Phase JP Nagar,3 BHK,2065.0,4.0,1.0,210.0,3,10169.491525,688.333333
3,Super built-up Area,Ready To Move,1st Phase JP Nagar,3 BHK,2059.0,3.0,2.0,225.0,3,10927.634774,686.333333
4,Super built-up Area,Ready To Move,1st Phase JP Nagar,2 BHK,1394.0,2.0,1.0,100.0,2,7173.601148,697.000000


In [2]:
#Select columns for RAG

rag_cols = [
    "area_type",
    "availability",
    "location",
    "total_sqft",
    "bath",
    "balcony",
    "price",
    "bhk"
]

rag_df = df[rag_cols].copy()
rag_df.head()

,area_type,availability,location,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,Ready To Move,1st Phase JP Nagar,1875.0,3.0,1.0,167.0,3
1,Built-up Area,Ready To Move,1st Phase JP Nagar,1500.0,5.0,2.0,85.0,5
2,Super built-up Area,Ready To Move,1st Phase JP Nagar,2065.0,4.0,1.0,210.0,3
3,Super built-up Area,Ready To Move,1st Phase JP Nagar,2059.0,3.0,2.0,225.0,3
4,Super built-up Area,Ready To Move,1st Phase JP Nagar,1394.0,2.0,1.0,100.0,2


In [3]:
#Create text for each property

rag_df["property_text"] = rag_df.apply(
    lambda row: f"""
    Property located in {row['location']}.
    Area type: {row['area_type']}.
    Availability: {row['availability']}.
    It has {int(row['bhk'])} BHK, {int(row['bath'])} bathrooms, and {int(row['balcony'])} balcony.
    Total area is {row['total_sqft']} square feet.
    Price is {row['price']} lakhs.
    """,
    axis=1
)

rag_df[["property_text"]].head()

,property_text
0,\n Property located in 1st Phase JP Nagar.\...
1,\n Property located in 1st Phase JP Nagar.\...
2,\n Property located in 1st Phase JP Nagar.\...
3,\n Property located in 1st Phase JP Nagar.\...
4,\n Property located in 1st Phase JP Nagar.\...


In [4]:
# Check one sample

print(rag_df["property_text"].iloc[0])


    Property located in 1st Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 1 balcony.
    Total area is 1875.0 square feet.
    Price is 167.0 lakhs.
    


Why convert rows to text?

SentenceTransformer operates on natural-language text. It cannot directly perform semantic search over an ordinary Pandas row.
***

## Generate Embeddings

In [5]:
!pip install sentence-transformers faiss-cpu

In [6]:
import sys
print(sys.version)
print(sys.executable)

3.10.20 (main, Jun 11 2026, 15:14:28) [Clang 20.1.8 ]
/opt/anaconda3/envs/houseprice_automl/bin/python


In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [8]:
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu

In [9]:
from sentence_transformers import SentenceTransformer

In [10]:
#load embeddings

In [11]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
documents = rag_df["property_text"].tolist()

embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/228 [00:00<?, ?it/s]

In [13]:
print(embeddings.shape)

(7296, 384)


***

# vectorDB - FAISS

In [14]:
import faiss

In [15]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

In [16]:
index.add(embeddings)

In [17]:
print(index.ntotal)

7296


In [18]:
query = """
I want a 3 BHK house in JP Nagar under 2 crore.
"""

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

distances, indices = index.search(
    query_embedding,
    k=5
)

In [19]:
for i, idx in enumerate(indices[0]):
    print(f"Rank {i+1}")
    print(f"Distance: {distances[0][i]:.4f}")
    print(rag_df.iloc[idx]["property_text"])
    print("-" * 80)

Rank 1
Distance: 0.5616

    Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 2 BHK, 2 bathrooms, and 1 balcony.
    Total area is 1460.0 square feet.
    Price is 80.0 lakhs.
    
--------------------------------------------------------------------------------
Rank 2
Distance: 0.5642

    Property located in 1st Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 2 BHK, 2 bathrooms, and 1 balcony.
    Total area is 900.0 square feet.
    Price is 75.0 lakhs.
    
--------------------------------------------------------------------------------
Rank 3
Distance: 0.5662

    Property located in JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 2 BHK, 2 bathrooms, and 3 balcony.
    Total area is 1200.0 square feet.
    Price is 78.0 lakhs.
    
--------------------------------------------------------------------------------
Rank 4
Di

In [20]:
filtered_df = rag_df[
    (rag_df["location"].str.contains("JP Nagar", case=False, na=False)) &
    (rag_df["bhk"] == 3) &
    (rag_df["price"] <= 200)
].copy()

filtered_df.shape

(76, 9)

In [21]:
filtered_df.head()

,area_type,availability,location,total_sqft,bath,balcony,price,bhk,property_text
0,Super built-up Area,Ready To Move,1st Phase JP Nagar,1875.0,3.0,1.0,167.0,3,\n Property located in 1st Phase JP Nagar.\...
9,Super built-up Area,18-May,1st Phase JP Nagar,1590.0,3.0,3.0,131.0,3,\n Property located in 1st Phase JP Nagar.\...
14,Super built-up Area,18-May,1st Phase JP Nagar,2077.0,3.0,3.0,175.0,3,\n Property located in 1st Phase JP Nagar.\...
29,Built-up Area,Ready To Move,5th Phase JP Nagar,1725.0,2.0,2.0,100.0,3,\n Property located in 5th Phase JP Nagar.\...
31,Built-up Area,Ready To Move,5th Phase JP Nagar,1700.0,2.0,3.0,100.0,3,\n Property located in 5th Phase JP Nagar.\...


In [22]:
filtered_documents = filtered_df["property_text"].tolist()

filtered_embeddings = embedding_model.encode(
    filtered_documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [23]:
dimension = filtered_embeddings.shape[1]

filtered_index = faiss.IndexFlatL2(dimension)
filtered_index.add(filtered_embeddings)

In [24]:
query = "I want a 3 BHK house in JP Nagar under 2 crore."

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

distances, indices = filtered_index.search(
    query_embedding,
    k=5
)

In [25]:
for i, idx in enumerate(indices[0]):
    print(f"Rank {i+1}")
    print(f"Distance: {distances[0][i]:.4f}")
    print(filtered_df.iloc[idx]["property_text"])
    print("-" * 80)

Rank 1
Distance: 0.5722

    Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 3 balcony.
    Total area is 2003.0 square feet.
    Price is 200.0 lakhs.
    
--------------------------------------------------------------------------------
Rank 2
Distance: 0.5810

    Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 2 balcony.
    Total area is 2040.0 square feet.
    Price is 200.0 lakhs.
    
--------------------------------------------------------------------------------
Rank 3
Distance: 0.5819

    Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 2 bathrooms, and 2 balcony.
    Total area is 1460.0 square feet.
    Price is 100.0 lakhs.
    
----------------------------------------------------------------------------

*** 

#### Pure FAISS semantic retrieval may not strictly satisfy numeric or categorical constraints such as BHK, budget, or location. Therefore, structured filtering is applied first, followed by FAISS-based semantic ranking on the filtered subset.

## Hybrid Retrieval Function

In [26]:
def hybrid_retrieve(query, location=None, bhk=None, max_price=None, k=5):
    temp_df = rag_df.copy()

    if location:
        temp_df = temp_df[
            temp_df["location"].str.contains(location, case=False, na=False)
        ]

    if bhk:
        temp_df = temp_df[temp_df["bhk"] == bhk]

    if max_price:
        temp_df = temp_df[temp_df["price"] <= max_price]

    if temp_df.empty:
        return []

    temp_documents = temp_df["property_text"].tolist()

    temp_embeddings = embedding_model.encode(
        temp_documents,
        convert_to_numpy=True
    )

    temp_index = faiss.IndexFlatL2(temp_embeddings.shape[1])
    temp_index.add(temp_embeddings)

    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    distances, indices = temp_index.search(
        query_embedding,
        k=min(k, len(temp_df))
    )

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i + 1,
            "distance": distances[0][i],
            "property_text": temp_df.iloc[idx]["property_text"]
        })

    return results

In [27]:
results = hybrid_retrieve(
    query="I want a 3 BHK house in JP Nagar under 2 crore",
    location="JP Nagar",
    bhk=3,
    max_price=200,
    k=5
)

for r in results:
    print(f"Rank {r['rank']}")
    print(f"Distance: {r['distance']:.4f}")
    print(r["property_text"])
    print("-" * 80)

Rank 1
Distance: 0.5703

    Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 3 balcony.
    Total area is 2003.0 square feet.
    Price is 200.0 lakhs.
    
--------------------------------------------------------------------------------
Rank 2
Distance: 0.5806

    Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 2 balcony.
    Total area is 2040.0 square feet.
    Price is 200.0 lakhs.
    
--------------------------------------------------------------------------------
Rank 3
Distance: 0.5816

    Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 2 bathrooms, and 2 balcony.
    Total area is 1460.0 square feet.
    Price is 100.0 lakhs.
    
----------------------------------------------------------------------------

- Copies the RAG DataFrame.
- Applies provided filters.
- Returns an empty result if nothing matches.
- Generates embeddings for the filtered records.
- Creates a filtered FAISS index.
- Embeds the user query.
- Searches for the top-K results.
- Returns ranked property descriptions and distances.

For the RAG component, I converted selected historical property records into natural-language descriptions and generated SentenceTransformer embeddings. I initially tested pure FAISS retrieval, but found that semantic similarity did not guarantee structured requirements such as location, BHK, and budget. Therefore, I implemented hybrid retrieval, which first applies structured filtering and then uses FAISS to rank eligible properties semantically.

*** 

## Groq LLM Integration

In [28]:
!{sys.executable} -m pip install groq python-dotenv

In [29]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

api_key = os.getenv("GROQ_API_KEY")

print(api_key[:10])

gsk_MuqEHt


In [30]:
from groq import Groq

client = Groq(api_key=api_key)

In [31]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response.choices[0].message.content)

Hello, how can I assist you today?


### Retrieval

In [32]:
retrieved_results = hybrid_retrieve(
    query="I want a 3 BHK house in JP Nagar under 2 crore",
    location="JP Nagar",
    bhk=3,
    max_price=200,
    k=5
)

In [33]:
retrieved_results

[{'rank': 1,
  'distance': 0.5703194,
  'property_text': '\n    Property located in 6th Phase JP Nagar.\n    Area type: Super built-up  Area.\n    Availability: Ready To Move.\n    It has 3 BHK, 3 bathrooms, and 3 balcony.\n    Total area is 2003.0 square feet.\n    Price is 200.0 lakhs.\n    '},
 {'rank': 2,
  'distance': 0.5805633,
  'property_text': '\n    Property located in 6th Phase JP Nagar.\n    Area type: Super built-up  Area.\n    Availability: Ready To Move.\n    It has 3 BHK, 3 bathrooms, and 2 balcony.\n    Total area is 2040.0 square feet.\n    Price is 200.0 lakhs.\n    '},
 {'rank': 3,
  'distance': 0.5816366,
  'property_text': '\n    Property located in 7th Phase JP Nagar.\n    Area type: Super built-up  Area.\n    Availability: Ready To Move.\n    It has 3 BHK, 2 bathrooms, and 2 balcony.\n    Total area is 1460.0 square feet.\n    Price is 100.0 lakhs.\n    '},
 {'rank': 4,
  'distance': 0.58241785,
  'property_text': '\n    Property located in 7th Phase JP Nagar.\n

### Create one context string

In [34]:
context_parts = []

for result in retrieved_results:
    context_parts.append(
        f"Property {result['rank']}:\n{result['property_text'].strip()}"
    )

retrieved_context = "\n\n".join(context_parts)

print(retrieved_context)

Property 1:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 3 balcony.
    Total area is 2003.0 square feet.
    Price is 200.0 lakhs.

Property 2:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 2 balcony.
    Total area is 2040.0 square feet.
    Price is 200.0 lakhs.

Property 3:
Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 2 bathrooms, and 2 balcony.
    Total area is 1460.0 square feet.
    Price is 100.0 lakhs.

Property 4:
Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 1 balcony.
    Total area is 1850.0 square feet.
    Price is 150.0 lakhs.

Property 5:
Property located in 7th Phase JP Nagar.
    Area type: S

### Construct the Groq prompt

In [35]:
user_query = "I want a 3 BHK house in JP Nagar under 2 crore."

prompt = f"""
You are a Bengaluru property recommendation assistant.

User requirement:
{user_query}

Retrieved properties:
{retrieved_context}

Instructions:
- Answer only using the retrieved properties above.
- Recommend the most relevant properties.
- Mention their location, area, bathrooms, balconies, and recorded price.
- Keep the prices in lakhs.
- Do not invent any property or price.
- Clearly state that these are historical records from the dataset.
"""

print(prompt)


You are a Bengaluru property recommendation assistant.

User requirement:
I want a 3 BHK house in JP Nagar under 2 crore.

Retrieved properties:
Property 1:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 3 balcony.
    Total area is 2003.0 square feet.
    Price is 200.0 lakhs.

Property 2:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 2 balcony.
    Total area is 2040.0 square feet.
    Price is 200.0 lakhs.

Property 3:
Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 2 bathrooms, and 2 balcony.
    Total area is 1460.0 square feet.
    Price is 100.0 lakhs.

Property 4:
Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, an

### Send the prompt to Groq

In [36]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a Bengaluru property recommendation assistant. "
                "Use only the retrieved property context. "
                "Do not invent properties, prices, or details."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

In [37]:
answer = response.choices[0].message.content

print(answer)

Based on your requirement of a 3 BHK house in JP Nagar under 2 crores, I recommend the following properties:

1. **Property 2**: Located in 6th Phase JP Nagar, this property has a super built-up area of 2040.0 square feet, 3 bathrooms, 2 balconies, and is priced at 200.0 lakhs. (Historical records from the dataset)

2. **Property 5**: Located in 7th Phase JP Nagar, this property has a super built-up area of 1850.0 square feet, 3 bathrooms, 2 balconies, and is priced at 150.0 lakhs. (Historical records from the dataset)

3. **Property 4**: Located in 7th Phase JP Nagar, this property has a super built-up area of 1850.0 square feet, 3 bathrooms, 1 balcony, and is priced at 150.0 lakhs. (Historical records from the dataset)

These properties meet your requirements and are available in JP Nagar. Please note that these are historical records from the dataset and may not reflect the current market situation.


In [38]:
# User query → Hybrid retrieval → Context → Prompt → Groq response
# User query
# → Structured filtering
# → FAISS semantic ranking
# → Retrieved context
# → Grounded prompt
# → Groq response

##### Architecture Clarification: RAG vs. Regression Model

###### 1. Responsibilities Breakdown
* **RAG Retrieval (Search):** Finds existing historical data matching user requirements (*"Which past properties match this?"*).
  * *Example:* Retrieves historical listings like `1850 sqft — ₹150 lakh`.
* **Regression Model (Prediction):** Calculates a brand-new estimated price based on input features (*"What should this specific property cost?"*).
  * *Example:* Predicts `₹148 lakh` for an 1800 sqft property in JP Nagar.

---

###### 2. Why Combine Both?
* **Regression Model alone:** Gives a prediction (`₹148 lakh`) without supporting context or evidence.
* **RAG alone:** Gives past historical prices (`₹100 lakh`, `₹150 lakh`, `₹200 lakh`) without calculating a custom estimate.
* **Combined Pipeline:** Passes both the **predicted price** and **historical comparables** into the **LLM (Groq)** to generate a contextual, well-explained recommendation.

---

##### 💡 Key Takeaway
* **Search/Recommendations only:** Need **RAG**
* **Price Prediction only:** Need **Regression Model**
* **Prediction + Explanation/Evidence:** Need **Both**

### Load the regression model

In [39]:
import pickle
import json

with open("../Models/house_price_model.pkl", "rb") as file:
    regression_model = pickle.load(file)

with open("../Models/model_columns.json", "r") as file:
    model_columns = json.load(file)

In [40]:
print(type(regression_model))
print("Number of model columns:", len(model_columns))
print("First 10 columns:", model_columns[:10])

<class 'sklearn.linear_model._base.LinearRegression'>
Number of model columns: 230
First 10 columns: ['total_sqft', 'bath', 'balcony', 'bhk', 'area_type_Carpet  Area', 'area_type_Plot  Area', 'area_type_Super built-up  Area', 'location_2nd Phase Judicial Layout', 'location_5th Phase JP Nagar', 'location_6th Phase JP Nagar']


### Create one model input row

In [41]:
input_data = pd.DataFrame(
    np.zeros((1, len(model_columns))),
    columns=model_columns
)

input_data.loc[0, ["total_sqft", "bath", "balcony", "bhk"]] = [
    1850,
    3,
    2,
    3
]

In [42]:
print("Input shape:", input_data.shape)

print(
    input_data[
        ["total_sqft", "bath", "balcony", "bhk"]
    ]
)

Input shape: (1, 230)
   total_sqft  bath  balcony  bhk
0      1850.0   3.0      2.0  3.0


### Set location and area type

In [43]:
location = "7th Phase JP Nagar"
area_type = "Super built-up  Area"

location_column = "location_" + location
area_type_column = "area_type_" + area_type

print("Location column:", location_column)
print("Location exists:", location_column in model_columns)

print("Area-type column:", area_type_column)
print("Area type exists:", area_type_column in model_columns)

Location column: location_7th Phase JP Nagar
Location exists: True
Area-type column: area_type_Super built-up  Area
Area type exists: True


In [44]:
input_data.loc[0, location_column] = 1
input_data.loc[0, area_type_column] = 1

print("Location value:", input_data.loc[0, location_column])
print("Area-type value:", input_data.loc[0, area_type_column])

Location value: 1.0
Area-type value: 1.0


### Generate the price prediction

In [45]:
prediction = regression_model.predict(input_data)

predicted_price = prediction[0]

print("Predicted price:", predicted_price, "lakhs")
print("Rounded predicted price:", round(predicted_price, 2), "lakhs")

Predicted price: 133.6262119845672 lakhs
Rounded predicted price: 133.63 lakhs


### Construct the combined prompt

In [46]:
combined_prompt = f"""
You are a Bengaluru house-price and property recommendation assistant.

User requirement:
{user_query}

Property submitted for price prediction:
- Location: {location}
- Area type: {area_type}
- Total area: 1850 square feet
- BHK: 3
- Bathrooms: 3
- Balconies: 2

Regression model prediction:
The estimated price is {predicted_price:.2f} lakhs.

Retrieved historical properties:
{retrieved_context}

Instructions:
- Clearly state that the regression value is an estimated price.
- Compare the predicted price with the retrieved historical property prices.
- Recommend only properties present in the retrieved context.
- Keep predicted and historical prices clearly separated.
- Do not invent any property, price, or feature.
- Mention that historical records may not represent current listings.
"""

print(combined_prompt)


You are a Bengaluru house-price and property recommendation assistant.

User requirement:
I want a 3 BHK house in JP Nagar under 2 crore.

Property submitted for price prediction:
- Location: 7th Phase JP Nagar
- Area type: Super built-up  Area
- Total area: 1850 square feet
- BHK: 3
- Bathrooms: 3
- Balconies: 2

Regression model prediction:
The estimated price is 133.63 lakhs.

Retrieved historical properties:
Property 1:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 3 balcony.
    Total area is 2003.0 square feet.
    Price is 200.0 lakhs.

Property 2:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 2 balcony.
    Total area is 2040.0 square feet.
    Price is 200.0 lakhs.

Property 3:
Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To M

In [47]:
# Regression prediction + RAG retrieval context

### Send the combined prompt to Groq

In [48]:
combined_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a grounded Bengaluru house-price assistant. "
                "Use only the regression prediction and retrieved properties "
                "provided in the prompt. Do not invent information."
            )
        },
        {
            "role": "user",
            "content": combined_prompt
        }
    ],
    temperature=0.2
)

In [49]:
final_answer = combined_response.choices[0].message.content

print(final_answer)


**Estimated House Price Prediction:**

The regression model predicts the estimated price of a 3 BHK house in 7th Phase JP Nagar to be **1.33 Crores (133.63 lakhs)**. Please note that this is an estimated price and may vary based on various factors.

**Retrieved Historical Properties:**

We have retrieved the following properties in JP Nagar that match your requirements:

1. **Property 3:** Located in 7th Phase JP Nagar, this property has 3 BHK, 2 bathrooms, and 2 balconies with a total area of 1460 square feet. The price of this property is **1.00 Crore (100.0 lakhs)**.
2. **Property 4:** Located in 7th Phase JP Nagar, this property has 3 BHK, 3 bathrooms, and 1 balcony with a total area of 1850 square feet. The price of this property is **1.50 Crores (150.0 lakhs)**.
3. **Property 5:** Located in 7th Phase JP Nagar, this property has 3 BHK, 3 bathrooms, and 2 balconies with a total area of 1850 square feet. The price of this property is **1.50 Crores (150.0 lakhs)**.

**Comparison of 

### Begin making prediction reusable

In [50]:
def predict_house_price(
    location,
    area_type,
    total_sqft,
    bath,
    balcony,
    bhk
):
    input_data = pd.DataFrame(
        np.zeros((1, len(model_columns))),
        columns=model_columns
    )

    input_data.loc[
        0,
        ["total_sqft", "bath", "balcony", "bhk"]
    ] = [
        total_sqft,
        bath,
        balcony,
        bhk
    ]

    location_column = "location_" + location
    area_type_column = "area_type_" + area_type

    if location_column in model_columns:
        input_data.loc[0, location_column] = 1
    else:
        input_data.loc[0, "location_other"] = 1

    if area_type_column in model_columns:
        input_data.loc[0, area_type_column] = 1
        
    predicted_price = regression_model.predict(input_data)[0]

    return float(predicted_price)

In [51]:
test_prediction = predict_house_price(
    location="7th Phase JP Nagar",
    area_type="Super built-up  Area",
    total_sqft=1850,
    bath=3,
    balcony=2,
    bhk=3
)

print("Predicted price:", round(test_prediction, 2), "lakhs")

Predicted price: 133.63 lakhs


In [52]:
# predict_house_price() → estimated price
# hybrid_retrieve()     → comparable properties

### Make context construction reusable

In [53]:
def format_retrieved_context(results):
    if not results:
        return "No matching historical properties were found."

    context_parts = []

    for result in results:
        context_parts.append(
            f"Property {result['rank']}:\n"
            f"{result['property_text'].strip()}"
        )

    return "\n\n".join(context_parts)

In [54]:
test_context = format_retrieved_context(retrieved_results)

print(test_context)

Property 1:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 3 balcony.
    Total area is 2003.0 square feet.
    Price is 200.0 lakhs.

Property 2:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 2 balcony.
    Total area is 2040.0 square feet.
    Price is 200.0 lakhs.

Property 3:
Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 2 bathrooms, and 2 balcony.
    Total area is 1460.0 square feet.
    Price is 100.0 lakhs.

Property 4:
Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 1 balcony.
    Total area is 1850.0 square feet.
    Price is 150.0 lakhs.

Property 5:
Property located in 7th Phase JP Nagar.
    Area type: S

### Make combined-prompt construction reusable

In [55]:
def build_combined_prompt(
    user_query,
    location,
    area_type,
    total_sqft,
    bath,
    balcony,
    bhk,
    predicted_price,
    retrieved_context
):
    return f"""
You are a Bengaluru house-price and property recommendation assistant.

User requirement:
{user_query}

Property submitted for price prediction:
- Location: {location}
- Area type: {area_type}
- Total area: {total_sqft} square feet
- BHK: {bhk}
- Bathrooms: {bath}
- Balconies: {balcony}

Regression model prediction:
The estimated price is {predicted_price:.2f} lakhs.

Retrieved historical properties:
{retrieved_context}

Instructions:
- Clearly state that the regression value is an estimated price.
- Call dataset prices "historical recorded prices", never "actual prices".
- Compare the estimate with the closest retrieved historical properties.
- The displayed properties already satisfy the location, BHK, and budget filters.
- If retrieved properties are displayed, do not say that no matching properties were found.
- Say that no matches were found only if the context explicitly says:
  "No matching historical properties were found."
- Recommend only properties present in the retrieved context.
- Do not invent a price range, listing, property, feature, or market information.
- Do not mention other properties outside the retrieved context.
- Keep predicted and historical prices clearly separated.
- Mention that the retrieved records are historical and may not be current listings.
"""

In [56]:
test_prompt = build_combined_prompt(
    user_query=user_query,
    location="7th Phase JP Nagar",
    area_type="Super built-up  Area",
    total_sqft=1850,
    bath=3,
    balcony=2,
    bhk=3,
    predicted_price=test_prediction,
    retrieved_context=test_context
)

print(test_prompt)


You are a Bengaluru house-price and property recommendation assistant.

User requirement:
I want a 3 BHK house in JP Nagar under 2 crore.

Property submitted for price prediction:
- Location: 7th Phase JP Nagar
- Area type: Super built-up  Area
- Total area: 1850 square feet
- BHK: 3
- Bathrooms: 3
- Balconies: 2

Regression model prediction:
The estimated price is 133.63 lakhs.

Retrieved historical properties:
Property 1:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 3 balcony.
    Total area is 2003.0 square feet.
    Price is 200.0 lakhs.

Property 2:
Property located in 6th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 2 balcony.
    Total area is 2040.0 square feet.
    Price is 200.0 lakhs.

Property 3:
Property located in 7th Phase JP Nagar.
    Area type: Super built-up  Area.
    Availability: Ready To M

### Make the Groq call reusable

In [57]:
def generate_groq_response(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a grounded Bengaluru house-price assistant. "
                    "Use only the regression prediction and retrieved properties "
                    "provided in the prompt. Do not invent information."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [58]:
test_answer = generate_groq_response(test_prompt)

print(test_answer)

Based on your requirements, I have estimated the price of a 3 BHK house in 7th Phase JP Nagar to be 133.63 lakhs. Please note that this is an estimated price and may not reflect the current market value.

I have retrieved historical recorded prices of properties in the same location that match your filters. Here are the properties that satisfy your requirements:

Property 1:
- Location: 6th Phase JP Nagar (Note: This property is not in 7th Phase JP Nagar, but it's in the same area)
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 3 balcony.
    Total area is 2003.0 square feet.
    Price is 200.0 lakhs.

Property 2:
- Location: 6th Phase JP Nagar (Note: This property is not in 7th Phase JP Nagar, but it's in the same area)
    Area type: Super built-up  Area.
    Availability: Ready To Move.
    It has 3 BHK, 3 bathrooms, and 2 balcony.
    Total area is 2040.0 square feet.
    Price is 200.0 lakhs.

Property 3:
- Location: 7th Ph

### Create the end-to-end function

In [59]:
def run_house_assistant(
    user_query,
    property_location,
    search_location,
    area_type,
    total_sqft,
    bath,
    balcony,
    bhk,
    max_price,
    k=5
):
    predicted_price = predict_house_price(
        location=property_location,
        area_type=area_type,
        total_sqft=total_sqft,
        bath=bath,
        balcony=balcony,
        bhk=bhk
    )

    retrieved_results = hybrid_retrieve(
        query=user_query,
        location=search_location,
        bhk=bhk,
        max_price=max_price,
        k=k
    )

    if not retrieved_results:
        return {
            "predicted_price": predicted_price,
            "retrieved_results": [],
            "final_answer": (
                f"The regression model estimates the property price at "
                f"₹{predicted_price:.2f} lakhs. However, no matching "
                f"historical properties were found for {search_location}, "
                f"{bhk} BHK, within a maximum budget of "
                f"₹{max_price:.2f} lakhs. Try increasing the budget "
                f"or broadening the location."
            )
        }

    retrieved_context = format_retrieved_context(
        retrieved_results
    )

    combined_prompt = build_combined_prompt(
        user_query=user_query,
        location=property_location,
        area_type=area_type,
        total_sqft=total_sqft,
        bath=bath,
        balcony=balcony,
        bhk=bhk,
        predicted_price=predicted_price,
        retrieved_context=retrieved_context
    )

    final_answer = generate_groq_response(
        combined_prompt
    )

    return {
        "predicted_price": predicted_price,
        "retrieved_results": retrieved_results,
        "final_answer": final_answer
    }

### Test the complete pipeline

In [60]:
assistant_result = run_house_assistant(
    user_query="I want a 3 BHK house in JP Nagar under 2 crore.",
    property_location="7th Phase JP Nagar",
    search_location="JP Nagar",
    area_type="Super built-up  Area",
    total_sqft=1850,
    bath=3,
    balcony=2,
    bhk=3,
    max_price=200,
    k=5
)

In [61]:
print(
    "Predicted price:",
    round(assistant_result["predicted_price"], 2),
    "lakhs"
)

print(
    "Number of retrieved properties:",
    len(assistant_result["retrieved_results"])
)

print("\nFinal assistant response:\n")
print(assistant_result["final_answer"])

Predicted price: 133.63 lakhs
Number of retrieved properties: 5

Final assistant response:

Based on your requirements, I have estimated the price of a 3 BHK house in 7th Phase JP Nagar to be 133.63 lakhs. Please note that this is an estimated price and may not reflect the current market value.

I have retrieved some historical properties that match your location and BHK requirements. Here are the properties:

1. Property located in 6th Phase JP Nagar:
    - Area type: Super built-up  Area.
    - Availability: Ready To Move.
    - It has 3 BHK, 3 bathrooms, and 3 balcony.
    - Total area is 2003.0 square feet.
    - Historical recorded price is 200.0 lakhs.

2. Property located in 6th Phase JP Nagar:
    - Area type: Super built-up  Area.
    - Availability: Ready To Move.
    - It has 3 BHK, 3 bathrooms, and 2 balcony.
    - Total area is 2040.0 square feet.
    - Historical recorded price is 200.0 lakhs.

3. Property located in 7th Phase JP Nagar:
    - Area type: Super built-up  Ar

In [62]:
# Prediction → Hybrid retrieval → Context → Prompt → Groq

### Test the no-match case

In [63]:
no_match_result = run_house_assistant(
    user_query="I want a 3 BHK house in JP Nagar under 20 lakhs.",
    property_location="7th Phase JP Nagar",
    search_location="JP Nagar",
    area_type="Super built-up  Area",
    total_sqft=1850,
    bath=3,
    balcony=2,
    bhk=3,
    max_price=20,
    k=5
)

In [64]:
print(
    "Retrieved properties:",
    len(no_match_result["retrieved_results"])
)

print("\nAssistant response:\n")
print(no_match_result["final_answer"])

Retrieved properties: 0

Assistant response:

The regression model estimates the property price at ₹133.63 lakhs. However, no matching historical properties were found for JP Nagar, 3 BHK, within a maximum budget of ₹20.00 lakhs. Try increasing the budget or broadening the location.
